In [1]:
import warnings
warnings.filterwarnings("ignore")

In [21]:
# setup envirn
import os
from pathlib import Path

cache_root = Path("/scratch/jindai/hf_cache")
cache_root.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(cache_root)

os.environ["HF_HUB_CACHE"] = str(cache_root / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(cache_root / "transformers")

print("HF_HOME =", os.environ["HF_HOME"])
print("HF_HUB_CACHE =", os.environ.get("HF_HUB_CACHE"))
print("TRANSFORMERS_CACHE =", os.environ.get("TRANSFORMERS_CACHE"))

HF_HOME = /scratch/jindai/hf_cache
HF_HUB_CACHE = /scratch/jindai/hf_cache/hub
TRANSFORMERS_CACHE = /scratch/jindai/hf_cache/transformers


In [75]:
import os, random, zipfile, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.notebook import tqdm
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt

from transformers import CLIPProcessor, CLIPModel
from safetensors.torch import load_file
from collections import defaultdict
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


CONFIG = {
    "base_seed": 171717,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),

    # trials
    "num_trials": 5,
    "trial_seed_stride": 1000,

    # extraction
    "batch_size": 32,

    # TF-IDF
    "tfidf_max_features": 512,

    # finetuned model folders in Drive
    "clip_finetuned_drive_dir": "clip_finetuned_softmax",
    "sigclip_finetuned_drive_dir": "sigmoidclip_finetuned_bce",
    "llama_sigclip_drive_dir": "llamasigclip_assets",

    "vae_ckpt_relpath": "multimodal_vae_400.pth",
    "llamavae_ckpt_relpath": "llamavae_text_vae.pth",
    
    # Fixed preference profiles
    "num_profiles": 1500,
    "interaction_k": 5,
    "pref_threshold": 0.2,

    # evaluation
    "top_k": 5,
    "kfold_splits": 5,
    "rec_std_mode": "profiles",
    
    "pinsage_num_walks": 50,
    "pinsage_walk_length": 10,
    "pinsage_top_m": 10,
    "pinsage_base_percentile": 90,
    
    #GATNE-I params
    "gatne_dim": 128,
    "gatne_epochs": 20,
    "gatne_lr": 1e-3,
    "gatne_wd": 1e-6,
    "gatne_batch": 8192,
    "gatne_num_neg": 10,

    #KNN
    "graph_knn_k": 30,
    "graph_sim_floor": 0.0,

}
DEVICE = CONFIG["device"]
print("Using device:", DEVICE)


# REPRODUCIBILITY HELPERS
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_global_seed(CONFIG["base_seed"])

Using device: cpu


In [76]:
import pandas as pd
import numpy as np

#paths
TRAIN_DF_PATH = "train_df_fixed_paths.csv"
TEST_DF_PATH  = "test_df_fixed_paths.csv"

FEATURE_ROOT = "features_pgl5"
FEATURE_KEY = "llamavae" # <=sigclip_ft / clip_ft / clip_base / tfidf / resnet50 llamasigclip / llamavae / vae_mu

TRAIN_IDS_PATH = f"{FEATURE_ROOT}/train_ids.npy"
TEST_IDS_PATH  = f"{FEATURE_ROOT}/test_ids.npy"

XTRAIN_PATH = f"{FEATURE_ROOT}/{FEATURE_KEY}/X_train.npy"
XTEST_PATH  = f"{FEATURE_ROOT}/{FEATURE_KEY}/X_test.npy"

#load dfs
train_df = pd.read_csv(TRAIN_DF_PATH)
test_df  = pd.read_csv(TEST_DF_PATH)

print("train_df:", train_df.shape)
print("test_df :", test_df.shape)

#load ids + features
train_ids = np.load(TRAIN_IDS_PATH, allow_pickle=True)
test_ids  = np.load(TEST_IDS_PATH,  allow_pickle=True)

X_train = np.load(XTRAIN_PATH)# (151, 512)
X_test  = np.load(XTEST_PATH)# (38, 512)

print("train_ids:", train_ids.shape, "X_train:", X_train.shape)
print("test_ids :", test_ids.shape,  "X_test :", X_test.shape)

assert "id" in train_df.columns and "id" in test_df.columns
assert len(train_ids) == X_train.shape[0]
assert len(test_ids)  == X_test.shape[0]
assert X_train.shape[1] == X_test.shape[1]

train_df = train_df.set_index("id").loc[train_ids].reset_index()
test_df  = test_df.set_index("id").loc[test_ids].reset_index()

assert (train_df["id"].values == train_ids).all()
assert (test_df["id"].values  == test_ids).all()

print("Loaded + aligned: df rows match feature order")
print("Embedding dim =", X_train.shape[1])

train_df: (151, 10)
test_df : (38, 10)
train_ids: (151,) X_train: (151, 768)
test_ids : (38,) X_test : (38, 768)
Loaded + aligned: df rows match feature order
Embedding dim = 768


In [77]:
# combind test and train to have full data
X_all = np.vstack([X_train, X_test]).astype(np.float32)
id_all = np.concatenate([train_ids, test_ids])

print("X_all:", X_all.shape, "id_all:", id_all.shape)

X_all: (189, 768) id_all: (189,)


In [78]:
import numpy as np

PREF_PATH = "user_pref_profiles.npz"   

data = np.load(PREF_PATH, allow_pickle=True)
interacted_ids = data["interacted_ids"]
preferred_mat  = data["preferred_mat"]
label_cols     = list(data["label_cols"])

print("Loaded profiles:", interacted_ids.shape, preferred_mat.shape, label_cols)

Loaded profiles: (1500, 5) (1500, 3) ['animal_label', 'myth_label', 'tree_label']


In [79]:
from tqdm.notebook import tqdm
from nltk.tokenize import word_tokenize

## PyGeometric Implementation from Saurabh code
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd


#GraphSAGE definition
class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels, project=True, normalize=False, aggr="max")
        self.conv2 = SAGEConv(hidden_channels, out_channels, project=True, normalize=False, aggr="max")

    def forward(self, x, edge_index):
        x = F.elu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

    def get_embeddings(self, x, edge_index):
        x = self.conv1(x, edge_index)
        return x

#Build graph from features
def build_graph(features_np, df_split, device=None):
    """
    Strict inductive user-item graph for one split (train or test).
    - nodes: [users | items]
    - edges: user <-> item from df_split['scroll_id']
    - user features: zeros
    - item features: features_np (must align with df_split row order)
    """
    if device is None:
        device = torch.device("cpu")

    users = df_split["scroll_id"].astype(str).values
    uniq_users = pd.unique(users)
    user_map = {sid: u for u, sid in enumerate(uniq_users)}
    user_id = np.array([user_map[s] for s in users], dtype=np.int64)

    num_users = len(uniq_users)
    num_items = len(df_split)
    d = features_np.shape[1]

    item_id = np.arange(num_items, dtype=np.int64)

    u_nodes = user_id
    i_nodes = num_users + item_id

    rows = np.concatenate([u_nodes, i_nodes])
    cols = np.concatenate([i_nodes, u_nodes])

    edge_index = torch.tensor([rows, cols], dtype=torch.long, device=device)

    # node features: [U zeros; I features]
    x_user = torch.zeros((num_users, d), dtype=torch.float32, device=device)
    x_item = torch.tensor(features_np, dtype=torch.float32, device=device)
    x = torch.cat([x_user, x_item], dim=0)

    data = Data(x=x, edge_index=edge_index)

    data.item_offset = num_users
    data.num_users = num_users
    data.num_items = num_items

    return data


# ---- Train GNN model ----
def train_gnn_model(model, data, labels, device, epochs=600):
    model = model.to(device)
    data = data.to(device)
    labels = labels.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    loss_fn = nn.CrossEntropyLoss()

    item_offset = int(data.item_offset)

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)          # (U+I, 2)
        out_items = out[item_offset:]                 # (I, 2)
        loss = loss_fn(out_items, labels)             # labels is (I,)
        loss.backward()
        optimizer.step()

    return model

In [80]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch

def evaluate_recommendation(
    model, data_train, data_test,
    df_train, df_test,
    interacted_ids, preferred_mat, label_cols,
    label_type,                 # "animal" / "myth" / "tree"
    top_k=5,
    seed=None, 
    n_eval_profiles=1500, 
    profile_sample_mode="subsample"

):
    """
        Profiles-based Precision@K aligned with Transductive site:
    - fixed interacted_ids / preferred_mat
    - user query = mean embedding of interacted train items
    - full ranking over all test items
    - relevance for a given label = test_item[label]==1
    - only evaluate profiles whose preferred_mat[:, label]==1
    Returns: (mean, std, n_profiles_used)
    """
    model.eval()

    # 1) get embeddings
    with torch.no_grad():
        emb_train_all = model.get_embeddings(data_train.x, data_train.edge_index).detach().cpu().numpy()
        emb_test_all  = model.get_embeddings(data_test.x,  data_test.edge_index).detach().cpu().numpy()

    emb_train = emb_train_all[int(data_train.item_offset):]   
    emb_test  = emb_test_all[int(data_test.item_offset):]    

    # 2) map label_type -> column index in preferred_mat
    label_col = f"{label_type}_label"
    label_map = {str(c).lower(): j for j, c in enumerate(label_cols)}
    if label_col.lower() not in label_map:
        raise ValueError(f"label_cols doesn't contain '{label_col}'. label_cols={label_cols}")
    pref_idx = label_map[label_col.lower()]

    # 3) test label vector (0/1)
    if label_col not in df_test.columns:
        raise ValueError(f"df_test missing column '{label_col}'")
    y_test = df_test[label_col].values.astype(int)  # shape (n_test,)

    # 4) id -> train index
    if "id" not in df_train.columns:
        raise ValueError("df_train must contain 'id' column for profile id mapping.")
    train_id_to_idx = {pid: i for i, pid in enumerate(df_train["id"].values)}
    
    candidate = np.where(preferred_mat[:, pref_idx] == 1)[0] 

    if profile_sample_mode == "all" or n_eval_profiles >= len(candidate):
        picked = candidate
    else:
        rng = np.random.default_rng(seed)
        picked = rng.choice(candidate, size=n_eval_profiles, replace=False)

        
    scores = []
    for p in picked:

        ids = interacted_ids[p]
        idxs = [train_id_to_idx[i] for i in ids if i in train_id_to_idx]
        if len(idxs) == 0:
            continue

        user_vec = emb_train[idxs].mean(axis=0, keepdims=True) 

        sims = cosine_similarity(user_vec, emb_test)[0]         
        top_idx = np.argsort(sims)[-top_k:][::-1]

        relevant = y_test[top_idx].sum()
        scores.append(relevant / top_k)

    if len(scores) == 0:
        return np.nan, np.nan, 0

    scores = np.array(scores, dtype=float)
    return float(scores.mean()), float(scores.std()), int(len(scores))


def evaluate_gnn_acc_binary(model, data_test, y_test_np, device):
    model.eval()
    data_test = data_test.to(device)
    item_offset = int(data_test.item_offset)

    with torch.no_grad():
        logits = model(data_test.x, data_test.edge_index)   
        logits_items = logits[item_offset:]                 
        preds = logits_items.argmax(dim=1).detach().cpu().numpy()

    return float((preds == y_test_np).mean())

In [81]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def vanilla_precision_at_k_profiles(
    E_train, E_test,
    df_train, df_test,
    interacted_ids, preferred_mat, label_cols,
    label_type,
    top_k=5,
    seed=None,
    n_eval_profiles=1500,
    profile_sample_mode="subsample" # "subsample" or "all"
):
    label_col = f"{label_type}_label"
    y_test = df_test[label_col].values.astype(int)

    # label_cols 
    label_map = {str(c).lower(): j for j, c in enumerate(label_cols)}
    pref_idx = label_map[label_col.lower()]
    
    candidate = np.where(preferred_mat[:, pref_idx] == 1)[0]

    if profile_sample_mode == "all" or n_eval_profiles >= len(candidate):
        picked = candidate
    else:
        rng = np.random.default_rng(seed)
        picked = rng.choice(candidate, size=n_eval_profiles, replace=False)

    # train id -> row index
    train_id_to_idx = {pid: i for i, pid in enumerate(df_train["id"].values)}

    scores = []
    for p in picked:
        ids = interacted_ids[p]
        idxs = [train_id_to_idx[i] for i in ids if i in train_id_to_idx]
        if len(idxs) == 0:
            continue

        user_vec = E_train[idxs].mean(axis=0, keepdims=True)
        sims = cosine_similarity(user_vec, E_test)[0]
        top_idx = np.argsort(sims)[-top_k:][::-1]
        scores.append(y_test[top_idx].sum() / top_k)

    if len(scores) == 0:
        return np.nan, np.nan, 0

    scores = np.array(scores, dtype=float)
    return float(scores.mean()), float(scores.std()), int(len(scores))


In [82]:
# for one label type
def evaluate_label_type(
    label_type,                 # "animal" / "myth" / "tree"
    train_df, test_df,
    X_train, X_test,
    device,
    interacted_ids, preferred_mat, label_cols,
    top_k=5,
    epochs=600,
    modality_name="FEATURE",
    seed=None
):
    # labels
    label_col = f"{label_type}_label"
    y_train_np = train_df[label_col].values.astype(int)
    y_test_np  = test_df[label_col].values.astype(int)
    y_train_t  = torch.tensor(y_train_np, dtype=torch.long)

    #Vanilla P@5
    v_mean, v_std, v_used = vanilla_precision_at_k_profiles(
        E_train=X_train,
        E_test=X_test,
        df_train=train_df,
        df_test=test_df,
        interacted_ids=interacted_ids,
        preferred_mat=preferred_mat,
        label_cols=label_cols,
        label_type=label_type,
        top_k=top_k,
        seed=seed,                          
        n_eval_profiles=CONFIG["num_profiles"],     
        profile_sample_mode="subsample",          
    
    #graphs (user-item)
    data_train = build_graph(X_train, train_df, device=device)
    data_test  = build_graph(X_test,  test_df,  device=device)


    #GraphSAGE train
    model = GraphSAGE(in_channels=X_train.shape[1], hidden_channels=12, out_channels=2)
    model = train_gnn_model(model, data_train, y_train_t, device, epochs=epochs)
    model.eval()

    #GraphSAGE Acc (on fixed test split)
    gnn_acc = evaluate_gnn_acc_binary(model, data_test, y_test_np, device)

    #Logistic Regression baseline
    skf = StratifiedKFold(
        n_splits=CONFIG["kfold_splits"],
        shuffle=True,
        random_state=seed,
    )

    fold_scores = []
    for tr_idx, te_idx in skf.split(X_train, y_train_np):
        lr = LogisticRegression(max_iter=2000)
        lr.fit(X_train[tr_idx], y_train_np[tr_idx])
        fold_scores.append(lr.score(X_train[te_idx], y_train_np[te_idx]))

    lr_acc = float(np.mean(fold_scores))



    #profiles-based P@5
    p_mean, p_std, n_used = evaluate_recommendation(
        model, data_train, data_test,
        train_df, test_df,
        interacted_ids, preferred_mat, label_cols,
        label_type=label_type,
        top_k=top_k,
        seed=seed,                          
        n_eval_profiles=CONFIG["num_profiles"],
        profile_sample_mode="subsample",
    )

    return {
        "Label": label_type.capitalize(),
        "Modality": modality_name,

        "GraphSAGE_Acc": f"{gnn_acc:.4f}",
        "RawFeat_Acc":   f"{lr_acc:.4f}",

        "RawFeat+GraphSAGE P@5": f"{p_mean:.4f} ± {p_std:.4f}",
        "RawFeat P@5":           f"{v_mean:.4f} ± {v_std:.4f}",

        "n_profiles_used (GNN/Vanilla)": f"{n_used}/{v_used}",
}

In [83]:
all_trial_rows = []

TRIAL_SEEDS = [CONFIG["base_seed"] + i * CONFIG["trial_seed_stride"] for i in range(CONFIG["num_trials"])]

for t, seed in enumerate(TRIAL_SEEDS):
    set_global_seed(seed)  

    rows = []
    for lt in ["animal", "myth", "tree"]:
        rows.append(
            evaluate_label_type(
                lt,
                train_df, test_df,
                X_train, X_test,
                DEVICE,
                interacted_ids, preferred_mat, label_cols,
                top_k=5,
                epochs=600,
                modality_name=FEATURE_KEY,
                seed=seed,
            )
        )

    df_trial = pd.DataFrame(rows)
    df_trial["trial"] = t
    df_trial["seed"] = seed
    all_trial_rows.append(df_trial)

trial_all_df = pd.concat(all_trial_rows, ignore_index=True)
trial_all_df

,Label,Modality,GraphSAGE_Acc,RawFeat_Acc,RawFeat+GraphSAGE P@5,RawFeat P@5,n_profiles_used (GNN/Vanilla),trial,seed
0,Animal,llamavae,0.5526,0.6292,0.5663 ± 0.2311,0.6227 ± 0.2014,1340/1340,0,171717
1,Myth,llamavae,0.6316,0.5561,0.5233 ± 0.1581,0.6843 ± 0.1049,1478/1478,0,171717
2,Tree,llamavae,0.6579,0.6557,0.2542 ± 0.0945,0.1273 ± 0.1052,1324/1324,0,171717
3,Animal,llamavae,0.5526,0.6292,0.6266 ± 0.2143,0.6227 ± 0.2014,1340/1340,1,172717
4,Myth,llamavae,0.6053,0.5430,0.5368 ± 0.1439,0.6843 ± 0.1049,1478/1478,1,172717
5,Tree,llamavae,0.6579,0.6557,0.2918 ± 0.1075,0.1273 ± 0.1052,1324/1324,1,172717
6,Animal,llamavae,0.5263,0.6226,0.5027 ± 0.2618,0.6227 ± 0.2014,1340/1340,2,173717
7,Myth,llamavae,0.6316,0.5430,0.5200 ± 0.1612,0.6843 ± 0.1049,1478/1478,2,173717
8,Tree,llamavae,0.6579,0.6557,0.2989 ± 0.1120,0.1273 ± 0.1052,1324/1324,2,173717
9,Animal,llamavae,0.5526,0.6292,0.5843 ± 0.1999,0.6227 ± 0.2014,1340/1340,3,174717


In [84]:
import numpy as np
import pandas as pd
import math

def parse_mean(s):
    if isinstance(s, str) and "±" in s:
        a, b = s.split("±")
        return float(a.strip()), float(b.strip())
    return float(s), 0.0

summary_rows = []
for label in ["Animal", "Myth", "Tree"]:
    sub = trial_all_df[trial_all_df["Label"] == label].copy()

    gnn_acc_vals = sub["GraphSAGE_Acc"].astype(float).values 
    raw_acc_vals = sub["RawFeat_Acc"].astype(float).values

    gnn_acc_mean = float(np.mean(gnn_acc_vals))
    gnn_acc_std  = float(np.std(gnn_acc_vals))

    raw_acc_mean = float(np.mean(raw_acc_vals))
    raw_acc_std  = float(np.std(raw_acc_vals))

    gnn_ms = np.array([parse_mean(x) for x in sub["RawFeat+GraphSAGE P@5"].values], dtype=float)
    raw_ms = np.array([parse_mean(x) for x in sub["RawFeat P@5"].values], dtype=float)

    n_used_str = sub["n_profiles_used (GNN/Vanilla)"].iloc[0]
    n_i = int(str(n_used_str).split("/")[0])

    means_g, stds_g = gnn_ms[:, 0], gnn_ms[:, 1]
    means_r, stds_r = raw_ms[:, 0], raw_ms[:, 1]

    gnn_p_mean = float(np.mean(means_g))
    gnn_p_var  = float(np.mean(stds_g**2 + means_g**2) - gnn_p_mean**2)
    gnn_p_std  = float(math.sqrt(max(gnn_p_var, 0.0)))

    raw_p_mean = float(np.mean(means_r))
    raw_p_var  = float(np.mean(stds_r**2 + means_r**2) - raw_p_mean**2)
    raw_p_std  = float(math.sqrt(max(raw_p_var, 0.0)))

    summary_rows.append({
        "Label": label,
        "Modality": sub["Modality"].iloc[0],

        "GraphSAGE_Acc": f"{gnn_acc_mean:.2f} ± {gnn_acc_std:.2f}",
        "RawFeat_Acc":   f"{raw_acc_mean:.2f} ± {raw_acc_std:.2f}",

        "RawFeat+GraphSAGE P@5": f"{gnn_p_mean:.2f} ± {gnn_p_std:.2f}",
        "RawFeat P@5":           f"{raw_p_mean:.2f} ± {raw_p_std:.2f}",

        "n_profiles_used (GNN/Vanilla)": n_used_str,
        "n_trials": len(sub),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Label,Modality,GraphSAGE_Acc,RawFeat_Acc,RawFeat+GraphSAGE P@5,RawFeat P@5,n_profiles_used (GNN/Vanilla),n_trials
0,Animal,llamavae,0.55 ± 0.01,0.62 ± 0.02,0.58 ± 0.24,0.62 ± 0.20,1340/1340,5
1,Myth,llamavae,0.63 ± 0.02,0.54 ± 0.01,0.52 ± 0.15,0.68 ± 0.10,1478/1478,5
2,Tree,llamavae,0.66 ± 0.00,0.66 ± 0.00,0.26 ± 0.12,0.13 ± 0.11,1324/1324,5


In [85]:
import os
import json
from pathlib import Path

OUT_DIR = Path("results_tables")
OUT_DIR.mkdir(parents=True, exist_ok=True)

run_tag = f"useritem_GraphSAGE_{FEATURE_KEY}_P{5}_E{600}_fixedsplit_fixedprofiles_trials{CONFIG['num_trials']}"

trial_csv = OUT_DIR / f"{run_tag}.trials.csv"
trial_pkl = OUT_DIR / f"{run_tag}.trials.pkl"
summary_csv = OUT_DIR / f"{run_tag}.summary.csv"
summary_pkl = OUT_DIR / f"{run_tag}.summary.pkl"

trial_all_df.to_csv(trial_csv, index=False)
trial_all_df.to_pickle(trial_pkl)

summary_df.to_csv(summary_csv, index=False)
summary_df.to_pickle(summary_pkl)

print("Saved:", trial_csv)
print("Saved:", trial_pkl)
print("Saved:", summary_csv)
print("Saved:", summary_pkl)

meta = {
    "run_tag": run_tag,
    "graph": "user-item",
    "model": "GraphSAGE",
    "feature_key": FEATURE_KEY,
    "modality_name": NAME_MAP.get(FEATURE_KEY, FEATURE_KEY) if "NAME_MAP" in globals() else FEATURE_KEY,
    "top_k": 5,
    "epochs": 600,
    "num_trials": int(CONFIG["num_trials"]),
    "trial_seeds": [int(CONFIG["base_seed"] + i * CONFIG["trial_seed_stride"]) for i in range(CONFIG["num_trials"])],
    "split": "fixed train/test from train_df_fixed_paths.csv + test_df_fixed_paths.csv",
    "profiles": "user_pref_profiles.npz (fixed)",
    "feature_root": str(FEATURE_ROOT) if "FEATURE_ROOT" in globals() else "features_pgl5",
    "saved_files": {
        "trials_csv": str(trial_csv),
        "trials_pkl": str(trial_pkl),
        "summary_csv": str(summary_csv),
        "summary_pkl": str(summary_pkl),
    }
}

meta_path = OUT_DIR / f"{run_tag}.meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

Saved: results_tables/useritem_GraphSAGE_llamavae_P5_E600_fixedsplit_fixedprofiles_trials5.trials.csv
Saved: results_tables/useritem_GraphSAGE_llamavae_P5_E600_fixedsplit_fixedprofiles_trials5.trials.pkl
Saved: results_tables/useritem_GraphSAGE_llamavae_P5_E600_fixedsplit_fixedprofiles_trials5.summary.csv
Saved: results_tables/useritem_GraphSAGE_llamavae_P5_E600_fixedsplit_fixedprofiles_trials5.summary.pkl
Saved meta: results_tables/useritem_GraphSAGE_llamavae_P5_E600_fixedsplit_fixedprofiles_trials5.meta.json


Pinsage

In [14]:
from collections import defaultdict, Counter

def build_ui_edge_index(df_split, user_mapping, num_users, device):
    item_local = np.arange(len(df_split), dtype=np.int64)
    user_ids = df_split["scroll_id"].astype(str).map(user_mapping).astype(np.int64).values

    u_nodes = user_ids
    i_nodes = num_users + item_local

    rows = np.concatenate([u_nodes, i_nodes])
    cols = np.concatenate([i_nodes, u_nodes])

    edge_index = torch.tensor([rows, cols], dtype=torch.long, device=device)
    return edge_index, len(df_split) 

def build_pinsage_item_graph_from_bipartite(
    edge_index_ui, num_users, num_items, X_items,
    num_walks=50, walk_length=5, top_m=20, seed=0, device=DEVICE
):
    src = edge_index_ui[0].detach().cpu().numpy()
    dst = edge_index_ui[1].detach().cpu().numpy()
    N = num_users + num_items
    adj = [[] for _ in range(N)]
    for s, d in zip(src, dst):
        adj[int(s)].append(int(d))

    rng = np.random.default_rng(seed)

    # item-item co-visitation counts
    co_counts = [defaultdict(int) for _ in range(num_items)]

    for i in range(num_items):
        start = num_users + i  # item node in bipartite graph
        if len(adj[start]) == 0:
            continue

        for _ in range(num_walks):
            cur = start
            for _h in range(walk_length):
                # item -> user
                if len(adj[cur]) == 0: break
                cur = rng.choice(adj[cur])
                # user -> item
                if len(adj[cur]) == 0: break
                cur = rng.choice(adj[cur])

                if cur >= num_users:
                    j = cur - num_users
                    if j != i:
                        co_counts[i][j] += 1

    # build item-item edges
    rows, cols = [], []
    for i in range(num_items):
        if len(co_counts[i]) == 0:
            continue
        top = sorted(co_counts[i].items(), key=lambda x: x[1], reverse=True)[:top_m]
        for j, _c in top:
            rows.append(i); cols.append(j)
            rows.append(j); cols.append(i)

    if len(rows) == 0:
        edge_index_item = torch.empty((2,0), dtype=torch.long, device=device)
    else:
        edge_index_item = torch.tensor([rows, cols], dtype=torch.long, device=device)

    data = Data(
        x=torch.tensor(X_items, dtype=torch.float32, device=device),
        edge_index=edge_index_item
    )
    data.item_offset = 0  
    return data

In [15]:
#for one label type
def evaluate_label_type_pinsage(
    label_type,                 # "animal" / "myth" / "tree"
    train_df, test_df,
    X_train, X_test,
    device,
    interacted_ids, preferred_mat, label_cols,
    top_k=5,
    epochs=600,
    modality_name="FEATURE", 
    rw_seed=None,
    seed=None
):
    # labels
    label_col = f"{label_type}_label"
    y_train_np = train_df[label_col].values.astype(int)
    y_test_np  = test_df[label_col].values.astype(int)
    y_train_t  = torch.tensor(y_train_np, dtype=torch.long)

    #Vanilla P@5
    v_mean, v_std, v_used = vanilla_precision_at_k_profiles(
        E_train=X_train,
        E_test=X_test,
        df_train=train_df,
        df_test=test_df,
        interacted_ids=interacted_ids,
        preferred_mat=preferred_mat,
        label_cols=label_cols,
        label_type=label_type,
        top_k=top_k,
        seed=seed,                         
        n_eval_profiles=CONFIG["num_profiles"],     
        profile_sample_mode="subsample",           
    )
    
    #graphs (item-item)
    edge_ui_train, n_train_items = build_ui_edge_index(train_df, user_mapping, num_users, device)
    edge_ui_test,  n_test_items  = build_ui_edge_index(test_df,  user_mapping, num_users, device)

    assert n_train_items == X_train.shape[0]
    assert n_test_items  == X_test.shape[0]

    data_train = build_pinsage_item_graph_from_bipartite(
        edge_ui_train, num_users, n_train_items, X_train,
        num_walks=50, walk_length=5, top_m=20, seed=seed, device=device
    )
    data_test = build_pinsage_item_graph_from_bipartite(
        edge_ui_test, num_users, n_test_items, X_test,
        num_walks=50, walk_length=5, top_m=20, seed=seed, device=device
    )


    #Train GNN on RW graph
    model = GraphSAGE(in_channels=X_train.shape[1], hidden_channels=12, out_channels=2)
    model = train_gnn_model(model, data_train, y_train_t, device, epochs=epochs)
    model.eval()

    #GraphSAGE Acc (on fixed test split)
    gnn_acc = evaluate_gnn_acc_binary(model, data_test, y_test_np, device)

    #Logistic Acc (raw features baseline)
    X_all = np.vstack([X_train, X_test]).astype(np.float32)
    
    y_all = np.concatenate([
        train_df[label_col].values.astype(int),
        test_df[label_col].values.astype(int)
    ])

    skf = StratifiedKFold(
        n_splits=CONFIG["kfold_splits"],
        shuffle=True,
        random_state=seed, 
    )

    fold_scores = []
    for tr_idx, te_idx in skf.split(X_all, y_all):
        lr = LogisticRegression(max_iter=2000)  
        lr.fit(X_all[tr_idx], y_all[tr_idx])
        fold_scores.append(lr.score(X_all[te_idx], y_all[te_idx]))

    lr_acc = float(np.mean(fold_scores))


    #profiles-based P@5
    p_mean, p_std, n_used = evaluate_recommendation(
        model, data_train, data_test,
        train_df, test_df,
        interacted_ids, preferred_mat, label_cols,
        label_type=label_type,
        top_k=top_k,
        seed=seed,                          
        n_eval_profiles=CONFIG["num_profiles"],
        profile_sample_mode="subsample",
    )

    return {
        "Label": label_type.capitalize(),
        "Modality": modality_name,
        "PinSAGE_Acc": f"{gnn_acc:.4f}",          
        "RawFeat_Acc": f"{lr_acc:.4f}",
        "RawFeat+PinSAGE P@5": f"{p_mean:.4f} ± {p_std:.4f}",
        "RawFeat P@5": f"{v_mean:.4f} ± {v_std:.4f}",
        "n_profiles_used (GNN/Vanilla)": f"{n_used}/{v_used}",
}



In [16]:
all_trial_rows_pinsage = []

TRIAL_SEEDS = [CONFIG["base_seed"] + i * CONFIG["trial_seed_stride"] for i in range(CONFIG["num_trials"])]

for t, seed in enumerate(TRIAL_SEEDS):
    set_global_seed(seed)

    rows = []
    for lt in ["animal", "myth", "tree"]:
        rows.append(
            evaluate_label_type_pinsage(
                lt,
                train_df, test_df,
                X_train, X_test,
                DEVICE,
                interacted_ids, preferred_mat, label_cols,
                top_k=5,
                epochs=600,
                modality_name=FEATURE_KEY,
                rw_seed=seed,
                seed=seed
            )
        )

    df_trial = pd.DataFrame(rows)
    df_trial["trial"] = t
    df_trial["seed"] = seed
    all_trial_rows_pinsage.append(df_trial)

trial_all_df_pinsage = pd.concat(all_trial_rows_pinsage, ignore_index=True)
trial_all_df_pinsage


,Label,Modality,PinSAGE_Acc,RawFeat_Acc,RawFeat+PinSAGE P@5,RawFeat P@5,n_profiles_used (GNN/Vanilla),trial,seed
0,Animal,vae_mu,0.5263,0.5550,0.6351 ± 0.1393,0.4899 ± 0.1864,1340/1340,0,171717
1,Myth,vae_mu,0.6053,0.5925,0.6055 ± 0.1259,0.6083 ± 0.1943,1478/1478,0,171717
2,Tree,vae_mu,0.4474,0.6669,0.3355 ± 0.2194,0.3014 ± 0.2026,1324/1324,0,171717
3,Animal,vae_mu,0.5263,0.5447,0.5482 ± 0.1357,0.4899 ± 0.1864,1340/1340,1,172717
4,Myth,vae_mu,0.5789,0.5871,0.7264 ± 0.1617,0.6083 ± 0.1943,1478/1478,1,172717
5,Tree,vae_mu,0.4737,0.6193,0.3669 ± 0.1993,0.3014 ± 0.2026,1324/1324,1,172717
6,Animal,vae_mu,0.5263,0.5027,0.5499 ± 0.1721,0.4899 ± 0.1864,1340/1340,2,173717
7,Myth,vae_mu,0.5526,0.6189,0.7426 ± 0.1283,0.6083 ± 0.1943,1478/1478,2,173717
8,Tree,vae_mu,0.4737,0.6767,0.3231 ± 0.2392,0.3014 ± 0.2026,1324/1324,2,173717
9,Animal,vae_mu,0.5263,0.5028,0.5252 ± 0.1395,0.4899 ± 0.1864,1340/1340,3,174717


In [17]:
import numpy as np
import pandas as pd

import numpy as np
import pandas as pd
import math

def parse_mean(s):
    if isinstance(s, str) and "±" in s:
        a, b = s.split("±")
        return float(a.strip()), float(b.strip())
    return float(s), 0.0

summary_rows = []
for label in ["Animal", "Myth", "Tree"]:
    sub = trial_all_df_pinsage[trial_all_df_pinsage["Label"] == label].copy()

    gnn_acc_vals = sub["PinSAGE_Acc"].astype(float).values
    raw_acc_vals = sub["RawFeat_Acc"].astype(float).values

    gnn_acc_mean = float(np.mean(gnn_acc_vals))
    gnn_acc_std  = float(np.std(gnn_acc_vals))

    raw_acc_mean = float(np.mean(raw_acc_vals))
    raw_acc_std  = float(np.std(raw_acc_vals))

    gnn_ms = np.array([parse_mean(x) for x in sub["RawFeat+PinSAGE P@5"].values], dtype=float)
    raw_ms = np.array([parse_mean(x) for x in sub["RawFeat P@5"].values], dtype=float)

    n_used_str = sub["n_profiles_used (GNN/Vanilla)"].iloc[0]
    n_i = int(str(n_used_str).split("/")[0])

    means_g, stds_g = gnn_ms[:, 0], gnn_ms[:, 1]
    means_r, stds_r = raw_ms[:, 0], raw_ms[:, 1]

    gnn_p_mean = float(np.mean(means_g))
    gnn_p_var  = float(np.mean(stds_g**2 + means_g**2) - gnn_p_mean**2)
    gnn_p_std  = float(math.sqrt(max(gnn_p_var, 0.0)))

    raw_p_mean = float(np.mean(means_r))
    raw_p_var  = float(np.mean(stds_r**2 + means_r**2) - raw_p_mean**2)
    raw_p_std  = float(math.sqrt(max(raw_p_var, 0.0)))

    summary_rows.append({
        "Label": label,
        "Modality": sub["Modality"].iloc[0],
        "PinSAGE_Acc": f"{gnn_acc_mean:.2f} ± {gnn_acc_std:.2f}",
        "RawFeat_Acc":   f"{raw_acc_mean:.2f} ± {raw_acc_std:.2f}",
        "RawFeat+PinSAGE P@5": f"{gnn_p_mean:.2f} ± {gnn_p_std:.2f}",
        "RawFeat P@5":           f"{raw_p_mean:.2f} ± {raw_p_std:.2f}",
        "n_profiles_used (GNN/Vanilla)": n_used_str,
        "n_trials": len(sub),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Label,Modality,PinSAGE_Acc,RawFeat_Acc,RawFeat+PinSAGE P@5,RawFeat P@5,n_profiles_used (GNN/Vanilla),n_trials
0,Animal,vae_mu,0.53 ± 0.00,0.53 ± 0.02,0.56 ± 0.16,0.49 ± 0.19,1340/1340,5
1,Myth,vae_mu,0.54 ± 0.05,0.60 ± 0.01,0.71 ± 0.15,0.61 ± 0.19,1478/1478,5
2,Tree,vae_mu,0.45 ± 0.03,0.65 ± 0.03,0.36 ± 0.21,0.30 ± 0.20,1324/1324,5


In [18]:
import json
from pathlib import Path

OUT_DIR = Path("results_tables")
OUT_DIR.mkdir(parents=True, exist_ok=True)

run_tag = f"useritem_PinSAGE_{FEATURE_KEY}_P{5}_E{600}_fixedsplit_fixedprofiles_trials{CONFIG['num_trials']}"

trial_csv = OUT_DIR / f"{run_tag}.trials.csv"
trial_pkl = OUT_DIR / f"{run_tag}.trials.pkl"
summary_csv = OUT_DIR / f"{run_tag}.summary.csv"
summary_pkl = OUT_DIR / f"{run_tag}.summary.pkl"

trial_all_df_pinsage.to_csv(trial_csv, index=False)
trial_all_df_pinsage.to_pickle(trial_pkl)

summary_df.to_csv(summary_csv, index=False)
summary_df.to_pickle(summary_pkl)

print("Saved:", trial_csv)
print("Saved:", trial_pkl)
print("Saved:", summary_csv)
print("Saved:", summary_pkl)

meta = {
    "run_tag": run_tag,
    "graph": "user-item",
    "model": "PinSAGE",
    "feature_key": FEATURE_KEY,
    "top_k": 5,
    "epochs": 600,
    "num_trials": int(CONFIG["num_trials"]),
    "trial_seeds": [int(CONFIG["base_seed"] + i * CONFIG["trial_seed_stride"]) for i in range(CONFIG["num_trials"])],
    "split": "fixed train/test from train_df_fixed_paths.csv + test_df_fixed_paths.csv",
    "profiles": "user_pref_profiles.npz (fixed)",
    "saved_files": {
        "trials_csv": str(trial_csv),
        "trials_pkl": str(trial_pkl),
        "summary_csv": str(summary_csv),
        "summary_pkl": str(summary_pkl),
    }
}

meta_path = OUT_DIR / f"{run_tag}.meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

Saved: results_tables/useritem_PinSAGE_vae_mu_P5_E600_fixedsplit_fixedprofiles_trials5.trials.csv
Saved: results_tables/useritem_PinSAGE_vae_mu_P5_E600_fixedsplit_fixedprofiles_trials5.trials.pkl
Saved: results_tables/useritem_PinSAGE_vae_mu_P5_E600_fixedsplit_fixedprofiles_trials5.summary.csv
Saved: results_tables/useritem_PinSAGE_vae_mu_P5_E600_fixedsplit_fixedprofiles_trials5.summary.pkl
Saved meta: results_tables/useritem_PinSAGE_vae_mu_P5_E600_fixedsplit_fixedprofiles_trials5.meta.json


GATNE-I

In [86]:
# GATNE-I (user-item) strict inductive
# train graph: (user-item edges) + (item-item KNN edges from features)
# Test item emb: inferred via train-item -> test-item attach (feature KNN)
import numpy as np
import pandas as pd
import math
import json
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity

def _l2_normalize_np(x, eps=1e-12):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(n, eps, None)

def build_useritem_train_edges(df_train):
    users = df_train["scroll_id"].astype(str).values
    uniq_users = pd.unique(users)
    user_map = {sid: u for u, sid in enumerate(uniq_users)}
    user_id = np.array([user_map[s] for s in users], dtype=np.int64)

    num_users = len(uniq_users)
    num_items = len(df_train)
    item_id = np.arange(num_items, dtype=np.int64)

    u_nodes = user_id
    i_nodes = num_users + item_id

    rows = np.concatenate([u_nodes, i_nodes])
    cols = np.concatenate([i_nodes, u_nodes])

    edge_index = torch.tensor([rows, cols], dtype=torch.long, device=DEVICE)
    return edge_index, user_map, num_users, num_items, (num_users + num_items)

def build_itemitem_knn_edges_trainitems(X_train, item_offset, k=30, sim_floor=0.0):
    """
    Build item-item kNN edges among TRAIN items only.
    Node ids are shifted by item_offset (which is num_users_train).
    Returns undirected edges within [item_offset .. item_offset+n_items-1]
    """
    Xtr = _l2_normalize_np(X_train.astype(np.float32))
    n_items = Xtr.shape[0]

    sim = cosine_similarity(Xtr, Xtr)
    np.fill_diagonal(sim, -1.0)  # exclude self
    if sim_floor > 0:
        sim[sim < sim_floor] = -1.0

    kk = min(k, max(n_items - 1, 1))
    idx = np.argpartition(-sim, kth=kk-1, axis=1)[:, :kk]  # (n_items, kk)

    src_item = np.repeat(np.arange(n_items), kk)
    dst_item = idx.reshape(-1)

    # shift to node ids in train graph
    src = item_offset + src_item
    dst = item_offset + dst_item

    # make undirected
    src_u = np.concatenate([src, dst], axis=0)
    dst_u = np.concatenate([dst, src], axis=0)

    edge_itemitem = torch.from_numpy(np.stack([src_u, dst_u], axis=0)).long().to(DEVICE)
    return edge_itemitem

def build_attach_trainitem_to_testitem(X_train, X_test, item_offset_train, n_train_nodes,
                                       k=30, sim_floor=0.0):
    """
    Build attach edges: TRAIN item nodes -> TEST item nodes (new nodes)
    src: train item node id in train graph: item_offset_train + train_item_idx
    dst: new test item node id: n_train_nodes + test_item_idx
    """
    Xtr = _l2_normalize_np(X_train.astype(np.float32))
    Xte = _l2_normalize_np(X_test.astype(np.float32))
    n_tr, n_te = Xtr.shape[0], Xte.shape[0]

    sim = cosine_similarity(Xte, Xtr)
    if sim_floor > 0:
        sim[sim < sim_floor] = -1.0

    kk = min(k, n_tr)
    idx = np.argpartition(-sim, kth=kk-1, axis=1)[:, :kk]

    src_train_item = idx.reshape(-1)  
    dst_test_item = np.repeat(np.arange(n_te), kk)  

    src = item_offset_train + src_train_item
    dst = n_train_nodes + dst_test_item

    edge_attach = torch.from_numpy(np.stack([src, dst], axis=0)).long().to(DEVICE)
    return edge_attach, kk

@torch.no_grad()
def infer_test_items_by_attach_mean(model, edge_attach, n_train_nodes, n_test_items, device):
    W = model.get_emb().detach()
    if W.device != device:
        W = W.to(device)

    src = edge_attach[0].to(device)
    dst = edge_attach[1].to(device) - n_train_nodes 

    D = W.size(1)
    out = torch.zeros((n_test_items, D), device=device)
    cnt = torch.zeros((n_test_items, 1), device=device)

    out.index_add_(0, dst, W[src])
    cnt.index_add_(0, dst, torch.ones((dst.size(0), 1), device=device))

    out = out / cnt.clamp_min(1.0)
    out = F.normalize(out, dim=1)

    return out.detach().cpu().numpy().astype(np.float32), cnt.squeeze(1).detach().cpu().numpy().astype(np.int64)

def gatne_useritem_schemeA_embeddings(X_train, X_test, train_df,
                                     knn_k, sim_floor,
                                     dim, epochs, lr, wd, batch, num_neg,
                                     seed):
    # 1) build train edges
    edge_ui, user_map_tr, n_users_tr, n_items_tr, n_nodes_tr = build_useritem_train_edges(train_df)
    edge_ii = build_itemitem_knn_edges_trainitems(
        X_train, item_offset=n_users_tr,
        k=knn_k, sim_floor=sim_floor
    )
    edge_train = torch.cat([edge_ui, edge_ii], dim=1)

    # 2) train GATNE on train nodes only
    model = train_gatne_one_type(
        edge_train,
        num_nodes_train=n_nodes_tr,
        device=DEVICE,
        emb_dim=dim,
        lr=lr,
        weight_decay=wd,
        epochs=epochs,
        batch_size=batch,
        num_neg=num_neg,
        seed=seed
    )

    # 3) slice train item embeddings
    E_train_all = model.get_emb().detach().cpu().numpy().astype(np.float32)
    E_train_items = E_train_all[n_users_tr:n_users_tr+n_items_tr]
    E_train_items = _l2_normalize_np(E_train_items)

    # 4) infer test item embeddings by attach mean (train items -> test items)
    edge_attach, kk = build_attach_trainitem_to_testitem(
        X_train, X_test,
        item_offset_train=n_users_tr,
        n_train_nodes=n_nodes_tr,
        k=knn_k, sim_floor=sim_floor
    )
    E_test_items, cnt = infer_test_items_by_attach_mean(model, edge_attach, n_nodes_tr, len(X_test), DEVICE)

    return E_train_items, E_test_items, cnt, {
        "n_users_tr": n_users_tr,
        "n_items_tr": n_items_tr,
        "n_nodes_tr": n_nodes_tr,
        "edges_ui": int(edge_ui.shape[1]),
        "edges_ii": int(edge_ii.shape[1]),
        "edges_total": int(edge_train.shape[1]),
        "attach_k": int(kk),
    }

def gatne_acc_from_item_emb(E_train_items, E_test_items, y_train, y_test):
    lr = LogisticRegression(max_iter=2000)
    lr.fit(E_train_items, y_train)
    pred = lr.predict(E_test_items)
    return float((pred == y_test).mean())

def evaluate_label_type_gatne_useritem_schemeA(label_type, seed):
    label_col = f"{label_type}_label"
    y_train = train_df[label_col].values.astype(int)
    y_test  = test_df[label_col].values.astype(int)

    # vanilla P@K
    v_mean, v_std, v_used = vanilla_precision_at_k_profiles(
        E_train=X_train,
        E_test=X_test,
        df_train=train_df,
        df_test=test_df,
        interacted_ids=interacted_ids,
        preferred_mat=preferred_mat,
        label_cols=label_cols,
        label_type=label_type,
        top_k=CONFIG["top_k"],
        seed=seed,
        n_eval_profiles=CONFIG["num_profiles"],
        profile_sample_mode="subsample",
    )

    # GATNE embeddings
    Etr_item, Ete_item, cnt_test, dbg = gatne_useritem_schemeA_embeddings(
        X_train, X_test, train_df,
        knn_k=int(CONFIG["graph_knn_k"]),
        sim_floor=float(CONFIG["graph_sim_floor"]),
        dim=int(CONFIG["gatne_dim"]),
        epochs=int(CONFIG["gatne_epochs"]),
        lr=float(CONFIG["gatne_lr"]),
        wd=float(CONFIG["gatne_wd"]),
        batch=int(CONFIG["gatne_batch"]),
        num_neg=int(CONFIG["gatne_num_neg"]),
        seed=seed
    )

    gatne_acc = gatne_acc_from_item_emb(Etr_item, Ete_item, y_train, y_test)

    skf = StratifiedKFold(
        n_splits=CONFIG["kfold_splits"],
        shuffle=True,
        random_state=seed,
    )
    fold_scores = []
    for tr_idx, te_idx in skf.split(X_train, y_train):
        lr0 = LogisticRegression(max_iter=2000)
        lr0.fit(X_train[tr_idx], y_train[tr_idx])
        fold_scores.append(lr0.score(X_train[te_idx], y_train[te_idx]))
    raw_acc = float(np.mean(fold_scores))

    p_mean, p_std, n_used = vanilla_precision_at_k_profiles(
        E_train=Etr_item,
        E_test=Ete_item,
        df_train=train_df,
        df_test=test_df,
        interacted_ids=interacted_ids,
        preferred_mat=preferred_mat,
        label_cols=label_cols,
        label_type=label_type,
        top_k=CONFIG["top_k"],
        seed=seed,
        n_eval_profiles=CONFIG["num_profiles"],
        profile_sample_mode="subsample",
    )

    n_test_items_used = int((cnt_test > 0).sum())

    return {
        "Label": label_type.capitalize(),
        "Modality": FEATURE_KEY,
        "GATNE_Acc": f"{gatne_acc:.4f}",
        "RawFeat_Acc": f"{raw_acc:.4f}",
        "RawFeat+GATNE P@5": f"{p_mean:.4f} ± {p_std:.4f}",
        "RawFeat P@5": f"{v_mean:.4f} ± {v_std:.4f}",
        "n_profiles_used (GATNE/Vanilla)": f"{n_used}/{v_used}",
        "n_test_items_used": n_test_items_used,
    }

In [87]:
TRIAL_SEEDS = [CONFIG["base_seed"] + i * CONFIG["trial_seed_stride"] for i in range(CONFIG["num_trials"])]

all_trial_rows_gatne = []
for t, seed in enumerate(TRIAL_SEEDS):
    set_global_seed(seed)
    rows = []
    for lt in ["animal", "myth", "tree"]:
        rows.append(evaluate_label_type_gatne_useritem_schemeA(lt, seed=seed))
    df_trial = pd.DataFrame(rows)
    df_trial["trial"] = t
    df_trial["seed"] = seed
    all_trial_rows_gatne.append(df_trial)

trial_all_df_gatne = pd.concat(all_trial_rows_gatne, ignore_index=True)
trial_all_df_gatne

,Label,Modality,GATNE_Acc,RawFeat_Acc,RawFeat+GATNE P@5,RawFeat P@5,n_profiles_used (GATNE/Vanilla),n_test_items_used,trial,seed
0,Animal,llamavae,0.5263,0.6292,0.5560 ± 0.2403,0.6227 ± 0.2014,1340/1340,38,0,171717
1,Myth,llamavae,0.5263,0.5561,0.5988 ± 0.2093,0.6843 ± 0.1049,1478/1478,38,0,171717
2,Tree,llamavae,0.6579,0.6557,0.3352 ± 0.2126,0.1273 ± 0.1052,1324/1324,38,0,171717
3,Animal,llamavae,0.5263,0.6292,0.5618 ± 0.2459,0.6227 ± 0.2014,1340/1340,38,1,172717
4,Myth,llamavae,0.6316,0.5430,0.6156 ± 0.1982,0.6843 ± 0.1049,1478/1478,38,1,172717
5,Tree,llamavae,0.6579,0.6557,0.3275 ± 0.2189,0.1273 ± 0.1052,1324/1324,38,1,172717
6,Animal,llamavae,0.5263,0.6226,0.5693 ± 0.2363,0.6227 ± 0.2014,1340/1340,38,2,173717
7,Myth,llamavae,0.5000,0.5430,0.6020 ± 0.1969,0.6843 ± 0.1049,1478/1478,38,2,173717
8,Tree,llamavae,0.6579,0.6557,0.3373 ± 0.2241,0.1273 ± 0.1052,1324/1324,38,2,173717
9,Animal,llamavae,0.5263,0.6292,0.5501 ± 0.2515,0.6227 ± 0.2014,1340/1340,38,3,174717


In [88]:
def parse_mean(s):
    if isinstance(s, str) and "±" in s:
        a, b = s.split("±")
        return float(a.strip()), float(b.strip())
    return float(s), 0.0

summary_rows = []
for label in ["Animal", "Myth", "Tree"]:
    sub = trial_all_df_gatne[trial_all_df_gatne["Label"] == label].copy()

    gatne_acc_vals = sub["GATNE_Acc"].astype(float).values
    raw_acc_vals   = sub["RawFeat_Acc"].astype(float).values

    gatne_acc_mean = float(np.mean(gatne_acc_vals))
    gatne_acc_std  = float(np.std(gatne_acc_vals))

    raw_acc_mean = float(np.mean(raw_acc_vals))
    raw_acc_std  = float(np.std(raw_acc_vals))

    gatne_ms = np.array([parse_mean(x) for x in sub["RawFeat+GATNE P@5"].values], dtype=float)
    raw_ms   = np.array([parse_mean(x) for x in sub["RawFeat P@5"].values], dtype=float)

    means_g, stds_g = gatne_ms[:, 0], gatne_ms[:, 1]
    means_r, stds_r = raw_ms[:, 0], raw_ms[:, 1]

    gatne_p_mean = float(np.mean(means_g))
    gatne_p_var  = float(np.mean(stds_g**2 + means_g**2) - gatne_p_mean**2)
    gatne_p_std  = float(math.sqrt(max(gatne_p_var, 0.0)))

    raw_p_mean = float(np.mean(means_r))
    raw_p_var  = float(np.mean(stds_r**2 + means_r**2) - raw_p_mean**2)
    raw_p_std  = float(math.sqrt(max(raw_p_var, 0.0)))

    n_used_str = sub["n_profiles_used (GATNE/Vanilla)"].iloc[0]

    summary_rows.append({
        "Label": label,
        "Modality": sub["Modality"].iloc[0],
        "GATNE_Acc": f"{gatne_acc_mean:.2f} ± {gatne_acc_std:.2f}",
        "RawFeat_Acc": f"{raw_acc_mean:.2f} ± {raw_acc_std:.2f}",
        "RawFeat+GATNE P@5": f"{gatne_p_mean:.2f} ± {gatne_p_std:.2f}",
        "RawFeat P@5": f"{raw_p_mean:.2f} ± {raw_p_std:.2f}",
        "n_profiles_used (GATNE/Vanilla)": n_used_str,
        "n_trials": len(sub),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Label,Modality,GATNE_Acc,RawFeat_Acc,RawFeat+GATNE P@5,RawFeat P@5,n_profiles_used (GATNE/Vanilla),n_trials
0,Animal,llamavae,0.53 ± 0.00,0.62 ± 0.02,0.56 ± 0.24,0.62 ± 0.20,1340/1340,5
1,Myth,llamavae,0.56 ± 0.06,0.54 ± 0.01,0.61 ± 0.20,0.68 ± 0.10,1478/1478,5
2,Tree,llamavae,0.66 ± 0.00,0.66 ± 0.00,0.33 ± 0.22,0.13 ± 0.11,1324/1324,5


In [89]:
OUT_DIR = Path("results_tables")
OUT_DIR.mkdir(parents=True, exist_ok=True)

KNN_K = int(CONFIG["graph_knn_k"])
SIM_FLOOR = float(CONFIG["graph_sim_floor"])
GATNE_DIM = int(CONFIG["gatne_dim"])
GATNE_EPOCHS = int(CONFIG["gatne_epochs"])
GATNE_NUM_NEG = int(CONFIG["gatne_num_neg"])
GATNE_LR = float(CONFIG["gatne_lr"])
GATNE_WD = float(CONFIG["gatne_wd"])
GATNE_BATCH = int(CONFIG["gatne_batch"])

run_tag = (
    f"useritem_GATNE1_schemeA_{FEATURE_KEY}"
    f"_k{KNN_K}_sim{SIM_FLOOR}"
    f"_dim{GATNE_DIM}_neg{GATNE_NUM_NEG}"
    f"_E{GATNE_EPOCHS}_trials{CONFIG['num_trials']}"
)

trial_csv   = OUT_DIR / f"{run_tag}.trials.csv"
trial_pkl   = OUT_DIR / f"{run_tag}.trials.pkl"
summary_csv = OUT_DIR / f"{run_tag}.summary.csv"
summary_pkl = OUT_DIR / f"{run_tag}.summary.pkl"

trial_all_df_gatne.to_csv(trial_csv, index=False)
trial_all_df_gatne.to_pickle(trial_pkl)
summary_df.to_csv(summary_csv, index=False)
summary_df.to_pickle(summary_pkl)

print("Saved:", trial_csv)
print("Saved:", trial_pkl)
print("Saved:", summary_csv)
print("Saved:", summary_pkl)

meta = {
    "run_tag": run_tag,
    "graph": "user-item + item-item(knn) (train-only)",
    "model": "GATNE-1type (scheme-A useritem)",
    "feature_key": FEATURE_KEY,
    "top_k": int(CONFIG["top_k"]),
    "gatne": {
        "dim": GATNE_DIM,
        "epochs": GATNE_EPOCHS,
        "num_neg": GATNE_NUM_NEG,
        "lr": GATNE_LR,
        "weight_decay": GATNE_WD,
        "batch_size": GATNE_BATCH,
    },
    "graph_build": {
        "knn_k": KNN_K,
        "sim_floor": SIM_FLOOR,
        "train_edges": "user-item edges from train_df + item-item knn edges among train items",
        "test_infer": "attach mean: train-item -> test-item knn",
    },
    "num_trials": int(CONFIG["num_trials"]),
    "trial_seeds": [int(s) for s in TRIAL_SEEDS],
    "split": "fixed train/test from train_df_fixed_paths.csv + test_df_fixed_paths.csv",
    "profiles": "user_pref_profiles.npz (fixed)",
    "saved_files": {
        "trials_csv": str(trial_csv),
        "trials_pkl": str(trial_pkl),
        "summary_csv": str(summary_csv),
        "summary_pkl": str(summary_pkl),
    }
}

meta_path = OUT_DIR / f"{run_tag}.meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

Saved: results_tables/useritem_GATNE1_schemeA_llamavae_k30_sim0.0_dim128_neg10_E20_trials5.trials.csv
Saved: results_tables/useritem_GATNE1_schemeA_llamavae_k30_sim0.0_dim128_neg10_E20_trials5.trials.pkl
Saved: results_tables/useritem_GATNE1_schemeA_llamavae_k30_sim0.0_dim128_neg10_E20_trials5.summary.csv
Saved: results_tables/useritem_GATNE1_schemeA_llamavae_k30_sim0.0_dim128_neg10_E20_trials5.summary.pkl
Saved meta: results_tables/useritem_GATNE1_schemeA_llamavae_k30_sim0.0_dim128_neg10_E20_trials5.meta.json
